In [ ]:
#Solo descomenta esta celda si quieres tu iniciar sesión personal en Google Colab

#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!pip install folium matplotlib mapclassify

# Geocodificación

In [ ]:
import geopandas as gpd
import pandas as pd
import shapely.geometry as Point
import os
import zipfile
import warnings
warnings.filterwarnings("ignore")

Cuando trabajamos con direcciones, es recomendable utilizar un archivo de texto plano (``archivo .txt``). Esto se debe a que las direcciones suelen incluir comas `,` para separar elementos como el nombre de la calle, número exterior, colonia o código postal. Si usamos un archivo `.csv`, estas comas podrían ser interpretadas como separadores de columnas, lo que provocaría errores en la estructura del archivo.

Por esta razón, una buena práctica es utilizar el punto y coma `;` como delimitador de columnas, lo que evita conflictos con las comas propias de las direcciones.

A continuación, se muestra un ejemplo del archivo `direcciones.txt`, que contiene una lista de direcciones que utilizaremos para el proceso de geocodificación:

![](../source/images/txt.png)

Carguemos el archivo `direcciones.txt` en un DataFrame.

In [ ]:
ruta = os.path.join('drive', 'MyDrive', 'tu carpeta del curso', 'datos')
df = pd.read_csv(os.path.join(ruta, 'direcciones.txt'), sep=';', encoding='utf-8', dtype=str)

Para aplicar la geocodificacion, utilizaremos la biblioteca `geopy`, que nos permite obtener las coordenadas geográficas (latitud y longitud) utilizando el servicio de geocodificación de _OpenStreetMap_.

En este caso, observamos que una de las direcciones **no pudo ser geocodificada correctamente**. ¿Por qué ocurre esto?

Generalmente, este tipo de errores se deben a problemas en la estructura o formato de la dirección. Puede tratarse de abreviaciones poco claras, errores ortográficos, o una combinación de elementos que el servicio de geocodificación no puede interpretar correctamente.

Para entender mejor el problema, vamos a utilizar el servicio [Nominatim](https://nominatim.openstreetmap.org/ui/search.html) y realizar algunas modificaciones en la dirección para analizar cómo responde el geocodificador.

Una vez que corregimos el texto en el archivo `direcciones.txt`, podemos proceder a cargarlo nuevamente en un DataFrame y aplicar la geocodificación.

Una vez que hayamos corregido la omisión, podemos guardar este conjunto de datos en cualquier formato, ya sea `.csv`, `.xlsx`, `.txt` o cualquier extensión espacial. En este caso, lo guardaremos en un formato `.kml` para visualizarlo en un mapa.

Recuerda que antes de guardar el archivo, tenemos que especificar el nombre de la columna que contiene los datos espaciales. Además, es importante indicar el sistema de referencia espacial (CRS) que estamos utilizando. En este caso, utilizaremos el CRS `EPSG:4326`.

A continuación, vamos a utilizar la herramienta de `MyMaps` de Google para visualizar de forma muy sencilla si las coordenadas geográficas obtenidas son correctas. Para ello, simplemente subimos el archivo `.kml` que hemos generado y lo visualizamos en el mapa.

## [MyMaps](https://www.google.com/intl/es/maps/about/mymaps/)

Una vez hechas las correcciones, veremos qué fue lo que cambió... Subimos el archivo `direcciones_corregidas.kml`

In [ ]:
geoc = gpd.read_file(os.path.join(ruta, 'direcciones-corregidas.kml'), encoding='utf-8')
geoc

Cargamos el segundo conjunto de datos `metropolitana.zip`

In [ ]:
ruta_extraccion = os.path.join(ruta, 'extracciones')

with zipfile.ZipFile(os.path.join(ruta, 'metropolitana.zip'), 'r') as zip_ref:
    zip_ref.extractall(ruta_extraccion)

In [ ]:
am = gpd.read_file(os.path.join(ruta_extraccion, 'metropolitana.shp'))
am

A continuación, podemos visualizar ambas capas (tanto puntos como polígonos):

# Spatial Join

## Relación: `intersects`

Selecciona todos los puntos que _tienen alguna intersección_ con los polígonos de la capa `am`, (incluso si solo rozan el borde).

## Relación: `within`
Selecciona solo los puntos que están completamente contenidos dentro del polígono.

## Relación: `contains`
Este caso se usa si tienes los puntos como capa de referencia y quieres saber qué polígonos contienen a esos puntos.

También podemos visualizar los resultados en un mapa interactivo:

In [ ]:
# Primer código (copiar)

# Análisis de vecinos más cercanos

In [ ]:
from shapely.ops import nearest_points
from shapely.geometry import LineString

Vamos a utilizar una capa vectorial de polígonos correspondiente a `Colonias`:

In [ ]:
colonias = gpd.read_file(os.path.join(ruta_extraccion, 'colonias_iieg', 'ISDC2020_2024.shp'))

In [ ]:
# ['Zapopan', 'Guadalajara', 'Tlajomulco de Zúñiga', 'El Salto', 'Juanacatlán', 'Ixtlahuacán de los Membrillos', 'Zapotlanejo', 'San Pedro Tlaquepaque', 'Acatlán de Juárez', 'Tonalá']

Como se observa en el gráfico anterior, las geometrías corresponden a `polígonos`. Sin embargo, para poder calcular distancias, necesitamos una referencia *puntual*. Por ello, obtendremos el _centroide_ de cada polígono.

In [ ]:
# Segundo código (copiar)

Filtremos solamente las columnas de nuestro interés...

In [ ]:
# ['ID','MUN_INEGI', 'GEOCOL', 'MUNICIPIO', 'ABR', 'NOMCOL1', 'CLASIFICAC', 'Tipo', 'CP', 'POBTOT', 'centroid', 'geometry']

El otro conjunto de datos... _¿observas algo raro?_

Corregimos...

Vamos a generar una función que nos permita ubicar los puntos más cercanos, utilizando `nearest_points`:

In [ ]:
# Tercer código (copiar)

Esta función permite encontrar el valor de un atributo (`value_column`) perteneciente al **objeto geográfico más cercano** a una geometría de entrada. 

`unary_union` crea una **geometría compuesta** (`MultiPoint`) del _GeoDataFrame_ de comparación. Pero veamos, específicamente qué nos arroja:

Lo anterior permite usar la función `nearest_points()`.

In [ ]:
# Cuarto código (copiar)